# TPU-Parallel HRM Baseline v2

This is a clean, restart-safe replacement for the original Phase 4+ notebook chain.

It keeps the fixed-carrier HRM experiment, but makes the run scientifically and operationally reproducible:

- one definition of every class and function;
- no hidden dependence on earlier failed cells;
- no `jax.checkpoint` in forward-only sweeps;
- dynamic coefficients, so same-shape candidates reuse one XLA compilation;
- clean targets are used only for evaluation, not guidance;
- every horizon regenerates a workload of the requested length;
- normalized recovery loss is comparable across horizons;
- deterministic config hashes, seed IDs, CSV and JSON exports;
- matched candidates within each phase and independent-seed validation afterward.

In Colab, select **Runtime → Change runtime type → TPU**, then run all cells. The notebook automatically uses a small smoke profile on non-TPU hardware and the full baseline profile on TPU.

## 0. Runtime and reproducibility profile

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import csv
import dataclasses
import hashlib
import json
import math
import time
from dataclasses import asdict, dataclass
from functools import partial
from pathlib import Path
from typing import NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
from jax import lax, random

BACKEND = jax.default_backend()
RUN_PROFILE = "tpu_baseline" if BACKEND == "tpu" else "smoke"

if RUN_PROFILE == "tpu_baseline":
    PROFILE = {
        "batch": 32,
        "height": 16,
        "width": 16,
        "channels": 8,
        "cognitive_dim": 32,
        "hierarchy_height": 8,
        "hierarchy_width": 8,
        "steps": 256,
        "seed_count": 5,
        "long_horizons": (1024, 2048),
    }
else:
    PROFILE = {
        "batch": 2,
        "height": 8,
        "width": 8,
        "channels": 4,
        "cognitive_dim": 16,
        "hierarchy_height": 4,
        "hierarchy_width": 4,
        "steps": 32,
        "seed_count": 2,
        "long_horizons": (64, 128),
    }

OUTPUT_DIR = Path.cwd() / "hrm_baseline_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("JAX version:", jax.__version__)
print("Backend:", BACKEND)
print("Devices:", jax.devices())
print("Run profile:", RUN_PROFILE)
print("Profile:", PROFILE)
print("Output directory:", OUTPUT_DIR)
if BACKEND != "tpu":
    print("NOTE: smoke profile selected. On a Colab TPU, the full baseline profile is selected automatically.")

JAX version: 0.7.2
Backend: cpu
Devices: [CpuDevice(id=0)]
Run profile: smoke
Profile: {'batch': 2, 'height': 8, 'width': 8, 'channels': 4, 'cognitive_dim': 16, 'hierarchy_height': 4, 'hierarchy_width': 4, 'steps': 32, 'seed_count': 2, 'long_horizons': (64, 128)}
Output directory: /content/hrm_baseline_outputs
NOTE: smoke profile selected. On a Colab TPU, the full baseline profile is selected automatically.


## 1. Configuration, state, and traceability

In [ ]:
@dataclass(frozen=True)
class ShapeConfig:
    batch: int
    height: int
    width: int
    channels: int
    cognitive_dim: int
    hierarchy_height: int
    hierarchy_width: int

    def validate(self):
        assert self.height % self.hierarchy_height == 0
        assert self.width % self.hierarchy_width == 0
        assert self.cognitive_dim >= self.channels
        assert min(asdict(self).values()) > 0
        return self


class Dynamics(NamedTuple):
    dt: jax.Array
    diffusion: jax.Array
    reaction_gain: jax.Array
    reaction_saturation: jax.Array
    input_gain: jax.Array
    memory_gain: jax.Array
    memory_decay: jax.Array
    cognitive_gain: jax.Array
    guidance_gain: jax.Array
    hierarchy_gain: jax.Array
    allocation_floor: jax.Array
    field_bound: jax.Array
    latent_bound: jax.Array


@dataclass(frozen=True)
class ExperimentConfig:
    shape: ShapeConfig
    steps: int = 256
    dt: float = 0.08
    diffusion: float = 0.18
    reaction_gain: float = 0.55
    reaction_saturation: float = 0.35
    input_gain: float = 0.35
    memory_gain: float = 0.18
    memory_decay: float = 0.08
    cognitive_gain: float = 0.10
    guidance_gain: float = 0.16
    hierarchy_gain: float = 0.12
    allocation_floor: float = 0.20
    field_bound: float = 4.0
    latent_bound: float = 4.0
    perturb_step: int = 128
    perturb_strength: float = 1.5
    recovery_threshold_fraction: float = 0.10
    recovery_weight: float = 0.25
    safety_weight: float = 0.05
    ledger_tolerance: float = 5e-4

    def replace(self, **changes):
        return dataclasses.replace(self, **changes)

    def validate(self):
        self.shape.validate()
        assert self.steps >= 4
        assert 0 <= self.perturb_step < self.steps
        assert self.dt >= 0
        assert self.field_bound > 0 and self.latent_bound > 0
        assert 0 <= self.allocation_floor <= 1
        return self

    def as_dict(self):
        return asdict(self)

    def config_hash(self):
        payload = json.dumps(self.as_dict(), sort_keys=True, separators=(",", ":"))
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()

    def run_id(self, seed=0, tag="run"):
        return f"{tag}-{self.config_hash()[:12]}-s{seed}"

    def dynamics(self):
        vals = (
            self.dt, self.diffusion, self.reaction_gain, self.reaction_saturation,
            self.input_gain, self.memory_gain, self.memory_decay, self.cognitive_gain,
            self.guidance_gain, self.hierarchy_gain, self.allocation_floor,
            self.field_bound, self.latent_bound,
        )
        return Dynamics(*(jnp.asarray(v, dtype=jnp.float32) for v in vals))


class State(NamedTuple):
    phi: jax.Array
    memory: jax.Array
    cognition: jax.Array
    hierarchy: jax.Array
    allocation: jax.Array


class Params(NamedTuple):
    field_to_cognition: jax.Array
    cognition_to_field: jax.Array
    hierarchy_mix: jax.Array
    predictor: jax.Array


class Inputs(NamedTuple):
    drive: jax.Array
    target: jax.Array
    perturb: jax.Array


class Ablation(NamedTuple):
    diffusion: float = 1.0
    reaction: float = 1.0
    memory: float = 1.0
    cognition: float = 1.0
    guidance: float = 1.0
    hierarchy: float = 1.0
    allocation: float = 1.0


class StepLedger(NamedTuple):
    input_norm: jax.Array
    diffusion_norm: jax.Array
    reaction_norm: jax.Array
    memory_norm: jax.Array
    cognition_norm: jax.Array
    guidance_norm: jax.Array
    hierarchy_norm: jax.Array
    perturb_norm: jax.Array
    correction_norm: jax.Array
    ledger_relative_error: jax.Array
    prediction_mse: jax.Array
    field_variance: jax.Array
    coherence: jax.Array
    active_fraction: jax.Array


SHAPE = ShapeConfig(
    batch=PROFILE["batch"],
    height=PROFILE["height"],
    width=PROFILE["width"],
    channels=PROFILE["channels"],
    cognitive_dim=PROFILE["cognitive_dim"],
    hierarchy_height=PROFILE["hierarchy_height"],
    hierarchy_width=PROFILE["hierarchy_width"],
).validate()

BASELINE_CONFIG = ExperimentConfig(
    shape=SHAPE,
    steps=PROFILE["steps"],
    perturb_step=PROFILE["steps"] // 2,
).validate()

FULL = Ablation()
print("Baseline config hash:", BASELINE_CONFIG.config_hash())
print(BASELINE_CONFIG)

Baseline config hash: ce1da88bcda09b457a411e14ef1457e0048e3ceb896b9370138ccb1301bf80a7
ExperimentConfig(shape=ShapeConfig(batch=2, height=8, width=8, channels=4, cognitive_dim=16, hierarchy_height=4, hierarchy_width=4), steps=32, dt=0.08, diffusion=0.18, reaction_gain=0.55, reaction_saturation=0.35, input_gain=0.35, memory_gain=0.18, memory_decay=0.08, cognitive_gain=0.1, guidance_gain=0.16, hierarchy_gain=0.12, allocation_floor=0.2, field_bound=4.0, latent_bound=4.0, perturb_step=16, perturb_strength=1.5, recovery_threshold_fraction=0.1, recovery_weight=0.25, safety_weight=0.05, ledger_tolerance=0.0005)


## 2. TPU-efficient operators and deterministic workload

In [ ]:
def laplacian_2d(x):
    return (
        4.0 * x
        - jnp.roll(x, 1, axis=1)
        - jnp.roll(x, -1, axis=1)
        - jnp.roll(x, 1, axis=2)
        - jnp.roll(x, -1, axis=2)
    )


def pool_to_hierarchy(x, shape):
    b, h, w, d = x.shape
    sh = h // shape.hierarchy_height
    sw = w // shape.hierarchy_width
    return x.reshape(
        b, shape.hierarchy_height, sh, shape.hierarchy_width, sw, d
    ).mean(axis=(2, 4))


def prolong_from_hierarchy(hx, shape):
    sh = shape.height // shape.hierarchy_height
    sw = shape.width // shape.hierarchy_width
    return jnp.repeat(jnp.repeat(hx, sh, axis=1), sw, axis=2)


def bounded(x, bound):
    return bound * jnp.tanh(x / bound)


def rms_batch(x):
    axes = tuple(range(1, x.ndim))
    return jnp.sqrt(jnp.mean(x * x, axis=axes) + 1e-12)


def make_params(key, shape):
    k1, k2, k3, k4 = random.split(key, 4)
    scale = 1.0 / jnp.sqrt(shape.channels)
    return Params(
        field_to_cognition=random.normal(
            k1, (shape.channels, shape.cognitive_dim), dtype=jnp.float32
        ) * scale,
        cognition_to_field=random.normal(
            k2, (shape.cognitive_dim, shape.channels), dtype=jnp.float32
        ) / jnp.sqrt(shape.cognitive_dim),
        hierarchy_mix=random.normal(
            k3, (shape.channels, shape.channels), dtype=jnp.float32
        ) * scale,
        predictor=random.normal(
            k4, (shape.cognitive_dim, shape.channels), dtype=jnp.float32
        ) / jnp.sqrt(shape.cognitive_dim),
    )


def make_initial_state(key, shape):
    k1, k2 = random.split(key)
    phi = 0.05 * random.normal(
        k1, (shape.batch, shape.height, shape.width, shape.channels), dtype=jnp.float32
    )
    memory = jnp.zeros_like(phi)
    cognition = 0.01 * random.normal(
        k2, (shape.batch, shape.cognitive_dim), dtype=jnp.float32
    )
    hierarchy = pool_to_hierarchy(phi, shape)
    allocation = jnp.ones((shape.batch, 1, 1, shape.channels), dtype=jnp.float32)
    return State(phi, memory, cognition, hierarchy, allocation)


def make_inputs(key, cfg):
    shape = cfg.shape
    noise_key, perturb_key = random.split(key)
    yy = jnp.linspace(-1.0, 1.0, shape.height, dtype=jnp.float32)
    xx = jnp.linspace(-1.0, 1.0, shape.width, dtype=jnp.float32)
    Y, X = jnp.meshgrid(yy, xx, indexing="ij")

    t = jnp.arange(cfg.steps, dtype=jnp.float32)[:, None]
    b = jnp.arange(shape.batch, dtype=jnp.float32)[None, :]
    phase = 2.0 * jnp.pi * (t / float(cfg.steps) + b / float(shape.batch))

    cx = 0.45 * jnp.sin(phase)
    cy = 0.45 * jnp.cos(phase * 1.37)
    sigma = 0.18 + 0.04 * jnp.sin(phase * 0.73)

    X4 = X[None, None, :, :]
    Y4 = Y[None, None, :, :]
    target_scalar = jnp.exp(
        -((X4 - cx[:, :, None, None]) ** 2 + (Y4 - cy[:, :, None, None]) ** 2)
        / (2.0 * sigma[:, :, None, None] ** 2)
    )
    channel_phase = jnp.linspace(
        0.7, 1.3, shape.channels, dtype=jnp.float32
    )[None, None, None, None, :]
    target = jnp.sin(channel_phase * jnp.pi * target_scalar[..., None])

    noise = 0.35 * random.normal(noise_key, target.shape, dtype=jnp.float32)
    drive = target + noise

    perturb = jnp.zeros_like(target)
    damage = cfg.perturb_strength * random.normal(
        perturb_key,
        (shape.batch, shape.height, shape.width, shape.channels),
        dtype=jnp.float32,
    )
    perturb = perturb.at[cfg.perturb_step].set(damage)
    return Inputs(drive=drive, target=target, perturb=perturb)


def seeded_components(seed, cfg):
    root = random.PRNGKey(seed)
    pkey, skey, ikey = random.split(root, 3)
    return (
        make_params(pkey, cfg.shape),
        make_initial_state(skey, cfg.shape),
        make_inputs(ikey, cfg),
    )

## 3. Attributed HRM transition and fused recurrent execution

In [ ]:
def hrm_step(state, inp, params, dyn, shape, ablation):
    drive_t, target_t, perturb_t = inp
    phi, memory, cognition, hierarchy, allocation = state

    pooled_phi = phi.mean(axis=(1, 2))
    predicted_channels = cognition @ params.predictor

    # No clean-target leakage: the active dynamics see only the noisy drive.
    observed_channels = drive_t.mean(axis=(1, 2))
    prediction_error = observed_channels - predicted_channels

    utility = jnp.abs(prediction_error) + 0.25 * jnp.mean(jnp.abs(phi), axis=(1, 2))
    priority = jax.nn.softmax(utility, axis=-1)
    priority = priority / (jnp.max(priority, axis=-1, keepdims=True) + 1e-8)
    soft_alloc = dyn.allocation_floor + (1.0 - dyn.allocation_floor) * priority
    next_allocation = (
        ablation.allocation * soft_alloc[:, None, None, :]
        + (1.0 - ablation.allocation) * jnp.ones_like(allocation)
    )

    a_input = dyn.dt * dyn.input_gain * drive_t * next_allocation
    a_diff = (
        -dyn.dt * dyn.diffusion * laplacian_2d(phi)
        * next_allocation * ablation.diffusion
    )
    reaction = dyn.reaction_gain * phi - dyn.reaction_saturation * phi ** 3
    a_react = dyn.dt * reaction * next_allocation * ablation.reaction
    a_memory = (
        dyn.dt * dyn.memory_gain * (memory - phi)
        * next_allocation * ablation.memory
    )
    cognitive_field = cognition @ params.cognition_to_field
    a_cognition = (
        dyn.dt * dyn.cognitive_gain * cognitive_field[:, None, None, :]
        * next_allocation * ablation.cognition
    )
    a_guidance = (
        dyn.dt * dyn.guidance_gain * jnp.tanh(prediction_error[:, None, None, :])
        * next_allocation * ablation.guidance
    )
    h_prolong = prolong_from_hierarchy(hierarchy, shape)
    h_mixed = jnp.einsum("bhwd,df->bhwf", h_prolong, params.hierarchy_mix)
    a_hierarchy = (
        dyn.dt * dyn.hierarchy_gain * (h_mixed - phi)
        * next_allocation * ablation.hierarchy
    )
    a_perturb = perturb_t

    proposed_phi = (
        phi + a_input + a_diff + a_react + a_memory + a_cognition
        + a_guidance + a_hierarchy + a_perturb
    )
    safe_phi = bounded(proposed_phi, dyn.field_bound)
    a_correction = safe_phi - proposed_phi

    next_memory = bounded(
        (1.0 - dyn.memory_decay) * memory + dyn.memory_decay * safe_phi,
        dyn.field_bound,
    )
    error_pad = jnp.pad(
        prediction_error,
        ((0, 0), (0, shape.cognitive_dim - shape.channels)),
    )
    cognitive_proposal = (
        cognition
        + dyn.dt * jnp.einsum("bd,dc->bc", pooled_phi, params.field_to_cognition)
        + dyn.dt * error_pad
    )
    next_cognition = bounded(cognitive_proposal, dyn.latent_bound)
    next_hierarchy = pool_to_hierarchy(safe_phi, shape)

    next_state = State(
        safe_phi, next_memory, next_cognition, next_hierarchy, next_allocation
    )

    observed_delta = safe_phi - phi
    reconstructed_delta = (
        a_input + a_diff + a_react + a_memory + a_cognition
        + a_guidance + a_hierarchy + a_perturb + a_correction
    )
    residual = observed_delta - reconstructed_delta
    rel_err = rms_batch(residual) / (rms_batch(observed_delta) + 1e-8)

    disagreement = 0.5 * (
        jnp.mean((safe_phi - jnp.roll(safe_phi, 1, axis=1)) ** 2, axis=(1, 2, 3))
        + jnp.mean((safe_phi - jnp.roll(safe_phi, 1, axis=2)) ** 2, axis=(1, 2, 3))
    )

    ledger = StepLedger(
        input_norm=rms_batch(a_input),
        diffusion_norm=rms_batch(a_diff),
        reaction_norm=rms_batch(a_react),
        memory_norm=rms_batch(a_memory),
        cognition_norm=rms_batch(a_cognition),
        guidance_norm=rms_batch(a_guidance),
        hierarchy_norm=rms_batch(a_hierarchy),
        perturb_norm=rms_batch(a_perturb),
        correction_norm=rms_batch(a_correction),
        ledger_relative_error=rel_err,
        prediction_mse=jnp.mean((safe_phi - target_t) ** 2, axis=(1, 2, 3)),
        field_variance=jnp.var(safe_phi, axis=(1, 2, 3)),
        coherence=1.0 / (1.0 + disagreement),
        active_fraction=jnp.mean(next_allocation, axis=(1, 2, 3)),
    )
    return next_state, ledger


@partial(jax.jit, static_argnames=("shape",))
def run_hrm(initial_state, inputs, params, dyn, shape, ablation=FULL):
    scan_inputs = (inputs.drive, inputs.target, inputs.perturb)

    def body(state, inp):
        return hrm_step(state, inp, params, dyn, shape, ablation)

    return lax.scan(body, initial_state, scan_inputs)


def block_until_ready(tree):
    return jax.tree_util.tree_map(
        lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x,
        tree,
    )


# Forward-only parameter sweeps do not benefit from rematerialization/checkpointing.
# Add jax.checkpoint only when a later experiment differentiates through the scan.

## 4. Metrics, safety checks, and matched experiment runner

In [ ]:
def mechanism_means(ledger):
    names = (
        "input", "diffusion", "reaction", "memory", "cognition",
        "guidance", "hierarchy", "perturb", "correction",
    )
    arrays = (
        ledger.input_norm, ledger.diffusion_norm, ledger.reaction_norm,
        ledger.memory_norm, ledger.cognition_norm, ledger.guidance_norm,
        ledger.hierarchy_norm, ledger.perturb_norm, ledger.correction_norm,
    )
    return {name: float(np.asarray(jnp.mean(arr))) for name, arr in zip(names, arrays)}


def score_ledger(ledger, final_state, cfg):
    mse_curve = np.asarray(jnp.mean(ledger.prediction_mse, axis=1), dtype=np.float64)
    t_steps = len(mse_curve)
    p = int(np.clip(cfg.perturb_step, 0, t_steps - 1))
    window = max(2, min(16, t_steps // 8))
    pre_start = max(0, p - window)
    pre = float(np.mean(mse_curve[pre_start:p])) if p > pre_start else float(mse_curve[p])

    if cfg.perturb_strength <= 0.0:
        peak = pre
        threshold = pre
        did_recover = True
        recovery_time = 0
        amplitude_reduction = 0.0
        l_recovery = 0.0
    else:
        response_end = min(t_steps, p + window)
        peak = float(np.max(mse_curve[p:response_end]))
        peak_delta = max(peak - pre, 1e-12)
        threshold = pre + cfg.recovery_threshold_fraction * peak_delta
        after = mse_curve[p:]
        recovered_indices = np.flatnonzero(after <= threshold)
        did_recover = bool(recovered_indices.size)
        recovery_time = int(recovered_indices[0]) if did_recover else int(len(after))
        final_level = float(np.mean(mse_curve[-window:]))
        amplitude_reduction = float(np.clip((peak - final_level) / peak_delta, 0.0, 1.0))
        recovery_fraction = recovery_time / max(len(after) - 1, 1)
        l_recovery = recovery_fraction + (1.0 - amplitude_reduction)
        if not did_recover:
            l_recovery += 1.0

    max_ledger_error = float(np.asarray(jnp.max(ledger.ledger_relative_error)))
    mean_correction = float(np.asarray(jnp.mean(ledger.correction_norm)))
    final_phi = np.asarray(final_state.phi)
    finite = bool(np.isfinite(final_phi).all() and np.isfinite(mse_curve).all())
    max_abs_phi = float(np.max(np.abs(final_phi)))
    bounded_pass = max_abs_phi <= cfg.field_bound + 1e-4
    ledger_pass = max_ledger_error <= cfg.ledger_tolerance

    l_task = float(np.mean(mse_curve))
    l_safety = mean_correction + 10.0 * max(max_ledger_error - cfg.ledger_tolerance, 0.0)
    if not finite:
        l_safety += 1000.0
    if not bounded_pass:
        l_safety += 100.0
    l_total = l_task + cfg.recovery_weight * l_recovery + cfg.safety_weight * l_safety

    return {
        "L_total": l_total,
        "L_task": l_task,
        "L_recovery": float(l_recovery),
        "L_safety": float(l_safety),
        "did_recover": did_recover,
        "recovery_time": recovery_time,
        "recovery_amplitude_reduction": amplitude_reduction,
        "pre_mse": pre,
        "peak_mse": peak,
        "recovery_threshold": threshold,
        "final_prediction_mse": float(mse_curve[-1]),
        "max_ledger_relative_error": max_ledger_error,
        "mean_coherence": float(np.asarray(jnp.mean(ledger.coherence))),
        "min_field_variance": float(np.asarray(jnp.min(ledger.field_variance))),
        "mean_active_fraction": float(np.asarray(jnp.mean(ledger.active_fraction))),
        "mean_correction_norm": mean_correction,
        "max_abs_phi": max_abs_phi,
        "finite": finite,
        "bounded_pass": bounded_pass,
        "ledger_pass": ledger_pass,
    }


ALL_RECORDS = []
RUN_ARTIFACTS = {}


def execute_config(phase, candidate, cfg, seed=0, ablation=FULL, keep_artifact=False):
    cfg.validate()
    params, initial, inputs = seeded_components(seed, cfg)
    t0 = time.perf_counter()
    final_state, ledger = run_hrm(
        initial, inputs, params, cfg.dynamics(), cfg.shape, ablation
    )
    block_until_ready((final_state, ledger))
    elapsed = time.perf_counter() - t0
    metrics = score_ledger(ledger, final_state, cfg)

    record = {
        "phase": phase,
        "candidate": candidate,
        "run_id": cfg.run_id(seed=seed, tag=phase.lower().replace(" ", "_")),
        "config_hash": cfg.config_hash(),
        "seed": seed,
        "backend": BACKEND,
        "profile": RUN_PROFILE,
        "steps": cfg.steps,
        "batch": cfg.shape.batch,
        "perturb_step": cfg.perturb_step,
        "perturb_strength": cfg.perturb_strength,
        "elapsed_seconds": elapsed,
        **metrics,
        **{f"mechanism_{k}": v for k, v in mechanism_means(ledger).items()},
    }
    ALL_RECORDS.append(record)
    if keep_artifact:
        RUN_ARTIFACTS[(phase, candidate, seed)] = (cfg, final_state, ledger)
    return record


def run_sweep(phase, candidates, seed=0, update_baseline=True):
    print(f"\n=== {phase} ===")
    phase_records = []
    for name, cfg in candidates.items():
        record = execute_config(phase, name, cfg, seed=seed)
        phase_records.append(record)
        print(
            f"{name:28s} L_total={record['L_total']:.6f} "
            f"task={record['L_task']:.6f} recovery={record['L_recovery']:.4f} "
            f"ledger={'PASS' if record['ledger_pass'] else 'FAIL'} "
            f"finite={record['finite']}"
        )
    eligible = [r for r in phase_records if r["finite"] and r["ledger_pass"] and r["bounded_pass"]]
    pool = eligible if eligible else phase_records
    best = min(pool, key=lambda r: r["L_total"])
    label = "Selected best" if update_baseline else "Lowest measured loss (evaluation only)"
    print(label + ":", best["candidate"], "L_total=", f"{best['L_total']:.6f}")
    best_cfg = candidates[best["candidate"]]
    return best_cfg if update_baseline else None, phase_records


# Compile and execute one matched baseline before sweeps.
print("Compiling and running initial baseline...")
BASELINE_RECORD = execute_config(
    "Phase 3 baseline", "active_baseline", BASELINE_CONFIG, seed=0, keep_artifact=True
)
print(json.dumps({k: BASELINE_RECORD[k] for k in (
    "L_total", "L_task", "L_recovery", "max_ledger_relative_error",
    "ledger_pass", "bounded_pass", "finite", "elapsed_seconds"
)}, indent=2))

Compiling and running initial baseline...
{
  "L_total": 0.88758101752601,
  "L_task": 0.3353546455036849,
  "L_recovery": 2.205950167239286,
  "max_ledger_relative_error": 9.921144373947755e-05,
  "ledger_pass": true,
  "bounded_pass": true,
  "finite": true,
  "elapsed_seconds": 1.0747909979999974
}


## Phase 4 — Reaction fine-tuning

In [ ]:
ACTIVE_BASELINE_CONFIG = BASELINE_CONFIG
reaction_candidates = {
    "R_A_low_reaction": ACTIVE_BASELINE_CONFIG.replace(reaction_gain=0.10, reaction_saturation=0.10),
    "R_B_baseline": ACTIVE_BASELINE_CONFIG,
    "R_C_high_gain": ACTIVE_BASELINE_CONFIG.replace(reaction_gain=0.90, reaction_saturation=0.35),
    "R_D_high_saturation": ACTIVE_BASELINE_CONFIG.replace(reaction_gain=0.55, reaction_saturation=0.55),
    "R_E_aggressive": ACTIVE_BASELINE_CONFIG.replace(reaction_gain=0.90, reaction_saturation=0.55),
}
ACTIVE_BASELINE_CONFIG, PHASE4_RECORDS = run_sweep(
    "Phase 4 reaction", reaction_candidates
)


=== Phase 4 reaction ===
R_A_low_reaction             L_total=0.579073 task=0.321088 recovery=1.0287 ledger=PASS finite=True
R_B_baseline                 L_total=0.887581 task=0.335355 recovery=2.2060 ledger=PASS finite=True
R_C_high_gain                L_total=0.975591 task=0.401815 recovery=2.2916 ledger=PASS finite=True
R_D_high_saturation          L_total=0.576637 task=0.297731 recovery=1.1131 ledger=PASS finite=True
R_E_aggressive               L_total=0.915714 task=0.352816 recovery=2.2487 ledger=PASS finite=True
Selected best: R_D_high_saturation L_total= 0.576637


## Phase 5 — Cognition fine-tuning

In [ ]:
cognition_candidates = {
    "C_A_low_gain": ACTIVE_BASELINE_CONFIG.replace(cognitive_gain=0.05),
    "C_B_baseline": ACTIVE_BASELINE_CONFIG,
    "C_C_high_gain": ACTIVE_BASELINE_CONFIG.replace(cognitive_gain=0.20),
    "C_D_very_high_gain": ACTIVE_BASELINE_CONFIG.replace(cognitive_gain=0.30),
}
ACTIVE_BASELINE_CONFIG, PHASE5_RECORDS = run_sweep(
    "Phase 5 cognition", cognition_candidates
)


=== Phase 5 cognition ===
C_A_low_gain                 L_total=0.576821 task=0.297858 recovery=1.1133 ledger=PASS finite=True
C_B_baseline                 L_total=0.576637 task=0.297731 recovery=1.1131 ledger=PASS finite=True
C_C_high_gain                L_total=0.576329 task=0.297520 recovery=1.1127 ledger=PASS finite=True
C_D_very_high_gain           L_total=0.576093 task=0.297357 recovery=1.1124 ledger=PASS finite=True
Selected best: C_D_very_high_gain L_total= 0.576093


## Phase 6 — Guidance fine-tuning without target leakage

In [ ]:
guidance_candidates = {
    "G_A_low_gain": ACTIVE_BASELINE_CONFIG.replace(guidance_gain=0.05),
    "G_B_baseline": ACTIVE_BASELINE_CONFIG,
    "G_C_high_gain": ACTIVE_BASELINE_CONFIG.replace(guidance_gain=0.30),
    "G_D_very_high_gain": ACTIVE_BASELINE_CONFIG.replace(guidance_gain=0.50),
}
ACTIVE_BASELINE_CONFIG, PHASE6_RECORDS = run_sweep(
    "Phase 6 guidance", guidance_candidates
)


=== Phase 6 guidance ===
G_A_low_gain                 L_total=0.576451 task=0.297797 recovery=1.1121 ledger=PASS finite=True
G_B_baseline                 L_total=0.576093 task=0.297357 recovery=1.1124 ledger=PASS finite=True
G_C_high_gain                L_total=0.576206 task=0.297298 recovery=1.1131 ledger=PASS finite=True
G_D_very_high_gain           L_total=0.577908 task=0.298541 recovery=1.1149 ledger=PASS finite=True
Selected best: G_B_baseline L_total= 0.576093


## Phase 7 — Joint local search

In [ ]:
c0 = ACTIVE_BASELINE_CONFIG.cognitive_gain
g0 = ACTIVE_BASELINE_CONFIG.guidance_gain
joint_candidates = {
    "J_baseline": ACTIVE_BASELINE_CONFIG,
    "J_cog_low": ACTIVE_BASELINE_CONFIG.replace(cognitive_gain=max(0.01, c0 * 0.8)),
    "J_cog_high": ACTIVE_BASELINE_CONFIG.replace(cognitive_gain=c0 * 1.2),
    "J_guidance_low": ACTIVE_BASELINE_CONFIG.replace(guidance_gain=max(0.01, g0 * 0.8)),
    "J_guidance_high": ACTIVE_BASELINE_CONFIG.replace(guidance_gain=g0 * 1.2),
    "J_both_low": ACTIVE_BASELINE_CONFIG.replace(
        cognitive_gain=max(0.01, c0 * 0.8), guidance_gain=max(0.01, g0 * 0.8)
    ),
    "J_both_high": ACTIVE_BASELINE_CONFIG.replace(
        cognitive_gain=c0 * 1.2, guidance_gain=g0 * 1.2
    ),
}
ACTIVE_BASELINE_CONFIG, PHASE7_RECORDS = run_sweep(
    "Phase 7 joint", joint_candidates
)


=== Phase 7 joint ===
J_baseline                   L_total=0.576093 task=0.297357 recovery=1.1124 ledger=PASS finite=True
J_cog_low                    L_total=0.576225 task=0.297449 recovery=1.1126 ledger=PASS finite=True
J_cog_high                   L_total=0.575987 task=0.297285 recovery=1.1123 ledger=PASS finite=True
J_guidance_low               L_total=0.576163 task=0.297456 recovery=1.1123 ledger=PASS finite=True
J_guidance_high              L_total=0.576050 task=0.297283 recovery=1.1125 ledger=PASS finite=True
J_both_low                   L_total=0.576271 task=0.297530 recovery=1.1124 ledger=PASS finite=True
J_both_high                  L_total=0.575919 task=0.297193 recovery=1.1124 ledger=PASS finite=True
Selected best: J_both_high L_total= 0.575919


## Phase 8 — Perturbation curriculum (evaluation only)

In [ ]:
perturbation_curriculum = {
    "P_none": ACTIVE_BASELINE_CONFIG.replace(perturb_strength=0.0),
    "P_low": ACTIVE_BASELINE_CONFIG.replace(perturb_strength=0.5),
    "P_medium": ACTIVE_BASELINE_CONFIG.replace(perturb_strength=1.5),
    "P_high": ACTIVE_BASELINE_CONFIG.replace(perturb_strength=3.0),
    "P_early_medium": ACTIVE_BASELINE_CONFIG.replace(
        perturb_strength=1.5, perturb_step=max(1, ACTIVE_BASELINE_CONFIG.steps // 4)
    ),
    "P_late_medium": ACTIVE_BASELINE_CONFIG.replace(
        perturb_strength=1.5, perturb_step=min(
            ACTIVE_BASELINE_CONFIG.steps - 2,
            3 * ACTIVE_BASELINE_CONFIG.steps // 4,
        )
    ),
}
_, PHASE8_RECORDS = run_sweep(
    "Phase 8 perturbation", perturbation_curriculum, update_baseline=False
)
# Keep a fixed medium perturbation for subsequent recovery validation.
ACTIVE_BASELINE_CONFIG = ACTIVE_BASELINE_CONFIG.replace(
    perturb_strength=1.5, perturb_step=ACTIVE_BASELINE_CONFIG.steps // 2
)


=== Phase 8 perturbation ===
P_none                       L_total=0.042611 task=0.042603 recovery=0.0000 ledger=PASS finite=True
P_low                        L_total=0.699107 task=0.106173 recovery=2.3714 ledger=PASS finite=True
P_medium                     L_total=0.575919 task=0.297193 recovery=1.1124 ledger=PASS finite=True
P_high                       L_total=0.608897 task=0.476070 recovery=0.5230 ledger=PASS finite=True
P_early_medium               L_total=0.518150 task=0.336189 recovery=0.7252 ledger=PASS finite=True
P_late_medium                L_total=0.830187 task=0.234203 recovery=2.3816 ledger=PASS finite=True
Lowest measured loss (evaluation only): P_none L_total= 0.042611


## Phases 9–10 — Recovery definition, ledger reconstruction, and safety gate

In [ ]:
RECOVERY_RECORD = execute_config(
    "Phase 9 recovery", "selected_medium_perturbation",
    ACTIVE_BASELINE_CONFIG, seed=0, keep_artifact=True
)

SAFETY_FIELDS = (
    "finite", "bounded_pass", "ledger_pass", "max_abs_phi",
    "max_ledger_relative_error", "mean_correction_norm",
    "did_recover", "recovery_time", "recovery_amplitude_reduction",
)
print("Phase 9 recovery report:")
print(json.dumps({k: RECOVERY_RECORD[k] for k in SAFETY_FIELDS}, indent=2))

activation_fields = [
    "mechanism_input", "mechanism_diffusion", "mechanism_reaction",
    "mechanism_memory", "mechanism_cognition", "mechanism_guidance",
    "mechanism_hierarchy", "mechanism_correction",
]
activation_pass = all(RECOVERY_RECORD[k] > 0.0 for k in activation_fields)
SAFETY_GATE = {
    "finite": RECOVERY_RECORD["finite"],
    "bounded": RECOVERY_RECORD["bounded_pass"],
    "ledger": RECOVERY_RECORD["ledger_pass"],
    "mechanisms_active": activation_pass,
}
SAFETY_GATE["passed"] = all(SAFETY_GATE.values())
print("Phase 10 safety gate:", SAFETY_GATE)
if not SAFETY_GATE["passed"]:
    raise RuntimeError(f"Safety gate failed: {SAFETY_GATE}")

Phase 9 recovery report:
{
  "finite": true,
  "bounded_pass": true,
  "ledger_pass": true,
  "max_abs_phi": 0.8730388879776001,
  "max_ledger_relative_error": 9.86438972176984e-05,
  "mean_correction_norm": 0.012650012969970703,
  "did_recover": true,
  "recovery_time": 15,
  "recovery_amplitude_reduction": 0.8876241828566949
}
Phase 10 safety gate: {'finite': True, 'bounded': True, 'ledger': True, 'mechanisms_active': True, 'passed': True}


## Phase 11 — Allocation controls

In [ ]:
allocation_candidates = {
    "A_no_floor": ACTIVE_BASELINE_CONFIG.replace(allocation_floor=0.0),
    "A_floor_010": ACTIVE_BASELINE_CONFIG.replace(allocation_floor=0.10),
    "A_baseline": ACTIVE_BASELINE_CONFIG,
    "A_floor_040": ACTIVE_BASELINE_CONFIG.replace(allocation_floor=0.40),
    "A_floor_060": ACTIVE_BASELINE_CONFIG.replace(allocation_floor=0.60),
}
ACTIVE_BASELINE_CONFIG, PHASE11_RECORDS = run_sweep(
    "Phase 11 allocation", allocation_candidates
)


=== Phase 11 allocation ===
A_no_floor                   L_total=0.578094 task=0.299010 recovery=1.1138 ledger=PASS finite=True
A_floor_010                  L_total=0.577005 task=0.298100 recovery=1.1131 ledger=PASS finite=True
A_baseline                   L_total=0.575919 task=0.297193 recovery=1.1124 ledger=PASS finite=True
A_floor_040                  L_total=0.573762 task=0.295387 recovery=1.1110 ledger=PASS finite=True
A_floor_060                  L_total=0.571623 task=0.293591 recovery=1.1096 ledger=PASS finite=True
Selected best: A_floor_060 L_total= 0.571623


## Phase 12 — Independent seeds

In [ ]:
INDEPENDENT_SEED_RECORDS = []
print("\n=== Phase 12 independent seeds ===")
for seed in range(PROFILE["seed_count"]):
    rec = execute_config(
        "Phase 12 independent seeds", f"seed_{seed}",
        ACTIVE_BASELINE_CONFIG, seed=seed
    )
    INDEPENDENT_SEED_RECORDS.append(rec)
    print(
        f"seed={seed} L_total={rec['L_total']:.6f} "
        f"recover={rec['did_recover']} time={rec['recovery_time']} "
        f"ledger={'PASS' if rec['ledger_pass'] else 'FAIL'}"
    )


def summarize_records(records, label):
    fields = (
        "L_total", "L_task", "L_recovery", "recovery_time",
        "recovery_amplitude_reduction", "max_ledger_relative_error",
        "mean_coherence", "mean_active_fraction",
    )
    summary = {"label": label, "count": len(records)}
    for field in fields:
        values = np.asarray([r[field] for r in records], dtype=np.float64)
        summary[f"{field}_mean"] = float(np.mean(values))
        summary[f"{field}_std"] = float(np.std(values))
    summary["recovery_rate"] = float(np.mean([r["did_recover"] for r in records]))
    summary["ledger_pass_rate"] = float(np.mean([r["ledger_pass"] for r in records]))
    summary["bounded_pass_rate"] = float(np.mean([r["bounded_pass"] for r in records]))
    return summary

SEED_SUMMARY = summarize_records(INDEPENDENT_SEED_RECORDS, "independent_seeds")
print(json.dumps(SEED_SUMMARY, indent=2))


=== Phase 12 independent seeds ===
seed=0 L_total=0.571623 recover=True time=15 ledger=PASS
seed=1 L_total=0.573738 recover=True time=15 ledger=PASS
{
  "label": "independent_seeds",
  "count": 2,
  "L_total_mean": 0.5726803682338667,
  "L_total_std": 0.0010578479778595717,
  "L_task_mean": 0.29502609540941194,
  "L_task_std": 0.0014346613897942007,
  "L_recovery_mean": 1.1081503053900739,
  "L_recovery_std": 0.0014725503334344081,
  "recovery_time_mean": 15.0,
  "recovery_time_std": 0.0,
  "recovery_amplitude_reduction_mean": 0.8918496946099261,
  "recovery_amplitude_reduction_std": 0.0014725503334343526,
  "max_ledger_relative_error_mean": 9.63904894888401e-05,
  "max_ledger_relative_error_std": 4.748580977320671e-07,
  "mean_coherence_mean": 0.7877464294433594,
  "mean_coherence_std": 0.0036745071411132812,
  "mean_active_fraction_mean": 0.9699243009090424,
  "mean_active_fraction_std": 0.002731889486312866,
  "recovery_rate": 1.0,
  "ledger_pass_rate": 1.0,
  "bounded_pass_rate": 

## Phase 13 — True long-horizon validation

In [ ]:
LONG_HORIZON_RECORDS = []
print("\n=== Phase 13 long horizons ===")
for horizon in PROFILE["long_horizons"]:
    horizon_cfg = ACTIVE_BASELINE_CONFIG.replace(
        steps=horizon,
        perturb_step=horizon // 2,
    ).validate()
    horizon_records = []
    for seed in range(PROFILE["seed_count"]):
        rec = execute_config(
            "Phase 13 long horizon", f"steps_{horizon}_seed_{seed}",
            horizon_cfg, seed=seed
        )
        horizon_records.append(rec)
        LONG_HORIZON_RECORDS.append(rec)
        print(
            f"steps={horizon:4d} seed={seed} L_total={rec['L_total']:.6f} "
            f"recover={rec['did_recover']} ledger={'PASS' if rec['ledger_pass'] else 'FAIL'}"
        )
    print(json.dumps(summarize_records(horizon_records, f"steps_{horizon}"), indent=2))


=== Phase 13 long horizons ===
steps=  64 seed=0 L_total=0.337318 recover=True ledger=PASS
steps=  64 seed=1 L_total=0.773363 recover=False ledger=PASS
{
  "label": "steps_64",
  "count": 2,
  "L_total_mean": 0.5553404579007021,
  "L_total_std": 0.21802287242560728,
  "L_task_mean": 0.21664504616637714,
  "L_task_std": 0.016091245488496497,
  "L_recovery_mean": 1.353223331602458,
  "L_recovery_std": 0.8076048043304714,
  "recovery_time_mean": 23.5,
  "recovery_time_std": 8.5,
  "recovery_amplitude_reduction_mean": 0.9048411845265742,
  "recovery_amplitude_reduction_std": 0.033411255943374685,
  "max_ledger_relative_error_mean": 9.103031698032282e-05,
  "max_ledger_relative_error_std": 2.9499751690309495e-06,
  "mean_coherence_mean": 0.8569832146167755,
  "mean_coherence_std": 0.000856250524520874,
  "mean_active_fraction_mean": 0.9410572052001953,
  "mean_active_fraction_std": 0.001738905906677246,
  "recovery_rate": 0.5,
  "ledger_pass_rate": 1.0,
  "bounded_pass_rate": 1.0
}
steps= 

## 14. Export baseline artifacts

In [ ]:
FINAL_CONFIG = ACTIVE_BASELINE_CONFIG
FINAL_SUMMARY = {
    "schema_version": "hrm-tpu-baseline-v2",
    "backend": BACKEND,
    "run_profile": RUN_PROFILE,
    "jax_version": jax.__version__,
    "final_config": FINAL_CONFIG.as_dict(),
    "final_config_hash": FINAL_CONFIG.config_hash(),
    "safety_gate": SAFETY_GATE,
    "independent_seed_summary": SEED_SUMMARY,
    "long_horizon_summaries": [
        summarize_records(
            [r for r in LONG_HORIZON_RECORDS if r["steps"] == horizon],
            f"steps_{horizon}",
        )
        for horizon in PROFILE["long_horizons"]
    ],
    "record_count": len(ALL_RECORDS),
}

json_path = OUTPUT_DIR / "hrm_tpu_baseline_summary.json"
csv_path = OUTPUT_DIR / "hrm_tpu_baseline_records.csv"
config_path = OUTPUT_DIR / "hrm_tpu_selected_config.json"

json_path.write_text(json.dumps(FINAL_SUMMARY, indent=2), encoding="utf-8")
config_path.write_text(json.dumps(FINAL_CONFIG.as_dict(), indent=2), encoding="utf-8")

fieldnames = sorted({key for record in ALL_RECORDS for key in record.keys()})
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(ALL_RECORDS)

print("Selected final config hash:", FINAL_CONFIG.config_hash())
print("Records:", len(ALL_RECORDS))
print("Saved:", json_path)
print("Saved:", csv_path)
print("Saved:", config_path)
print(json.dumps(FINAL_SUMMARY, indent=2))

Selected final config hash: b1c699ca9bd45ed5d9bd352355343a187c86b5ac55051b1386f2f40abb4dc94a
Records: 39
Saved: /content/hrm_baseline_outputs/hrm_tpu_baseline_summary.json
Saved: /content/hrm_baseline_outputs/hrm_tpu_baseline_records.csv
Saved: /content/hrm_baseline_outputs/hrm_tpu_selected_config.json
{
  "schema_version": "hrm-tpu-baseline-v2",
  "backend": "cpu",
  "run_profile": "smoke",
  "jax_version": "0.7.2",
  "final_config": {
    "shape": {
      "batch": 2,
      "height": 8,
      "width": 8,
      "channels": 4,
      "cognitive_dim": 16,
      "hierarchy_height": 4,
      "hierarchy_width": 4
    },
    "steps": 32,
    "dt": 0.08,
    "diffusion": 0.18,
    "reaction_gain": 0.55,
    "reaction_saturation": 0.55,
    "input_gain": 0.35,
    "memory_gain": 0.18,
    "memory_decay": 0.08,
    "cognitive_gain": 0.36,
    "guidance_gain": 0.192,
    "hierarchy_gain": 0.12,
    "allocation_floor": 0.6,
    "field_bound": 4.0,
    "latent_bound": 4.0,
    "perturb_step": 16,
 

## Interpretation boundary

A successful run establishes a reproducible computational baseline for this fixed-carrier HRM prototype: bounded execution, active mechanisms, exact-enough transition attribution, matched local parameter searches, perturbation recovery metrics, independent-seed variability, and true long-horizon behavior.

It does **not** by itself establish a novel computing paradigm, general reasoning, dynamic topology, hardware advantage, or superiority over matched conventional baselines. Those require separate experiments and controls.